# Tick pipeline — Colab (3a + Block 4 combination critic)

Live `src/med_doc` on branch **`block1`**. Opening from GitHub does **not** ship `src/` — run the first cell until it prints **`BOOTSTRAP_V7`** and **`combo_backend`**.

**This notebook**
- Block 1 normalize
- Block 3 **`htr_mode="nonverbal"`** (ticks only; skip handwriting / 3b)
- 3c vision **off** unless you set `VISION_BACKEND` (needs local Ollama; Colab usually does not)
- Block 4 **`combo_backend="kg"`**: placeholder co-occurrence (profile components missing from the tick set → review). Swap to `"ollama"` for the small Instruct placeholder.
- Combination flags **≥ 0.55** re-run that checkbox crop through 3a and go to HiTL. They do **not** auto-write LIS ticks.

Do **not** upload clinic PHI. Default input is a **synthetic** form with slashes on `cbc` and `profile_lipid`.

[Older full 1–5 notebook](https://colab.research.google.com/github/RwaRwa599/epq3/blob/block1/Pipeline_Blocks_1_to_5.ipynb)


## 0. Download src/med_doc (zipball)


In [ ]:
# BOOTSTRAP_V7 — zipball of branch block1 (combo critic + htr_mode)
import importlib
import inspect
import os
import shutil
import sys
import urllib.request
import zipfile
from pathlib import Path

CONTENT = Path("/content") if Path("/content").is_dir() else Path.cwd()
REPO = CONTENT / "epq3"
SRC = REPO / "src"
URL = "https://codeload.github.com/RwaRwa599/epq3/zip/refs/heads/block1"

zpath = CONTENT / "epq3-block1.zip"
print("Downloading", URL)
urllib.request.urlretrieve(URL, zpath)
extract = CONTENT / "_epq3_extract"
if extract.exists():
    shutil.rmtree(extract)
extract.mkdir()
with zipfile.ZipFile(zpath) as zf:
    zf.extractall(extract)
found = list(extract.glob("*/src/med_doc/__init__.py"))
if not found:
    raise RuntimeError(f"zip missing src/med_doc: {list(extract.iterdir())}")
unpacked = found[0].parents[2]
if REPO.exists():
    shutil.rmtree(REPO)
shutil.move(str(unpacked), str(REPO))
shutil.rmtree(extract, ignore_errors=True)
zpath.unlink(missing_ok=True)

src = str(SRC.resolve())
while src in sys.path:
    sys.path.remove(src)
sys.path.insert(0, src)
os.chdir(REPO)
for name in list(sys.modules):
    if name == "med_doc" or name.startswith("med_doc."):
        del sys.modules[name]
importlib.invalidate_caches()
import med_doc
from med_doc.pipeline import run_blocks_1_to_5

print("BOOTSTRAP_V7")
print("cwd:", os.getcwd())
print("med_doc:", med_doc.__file__)
print("params:", list(inspect.signature(run_blocks_1_to_5).parameters))
from med_doc.htr.marks import TICK_POLICY
print("tick_policy:", TICK_POLICY)
sig = inspect.signature(run_blocks_1_to_5).parameters
if "combo_backend" not in sig or "htr_mode" not in sig:
    raise RuntimeError(
        "stale med_doc (need combo_backend and htr_mode). "
        "Runtime → Disconnect and delete runtime, re-open Tick_Pipeline_Colab.ipynb from branch block1."
    )
if TICK_POLICY != "slash-v2":
    raise RuntimeError(f"stale tick_policy={TICK_POLICY!r}")


In [ ]:
import subprocess
import sys

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless", "pydantic", "matplotlib", "Pillow", "numpy"]
)
print("pip ok")


In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    content = Path("/content") if Path("/content").is_dir() else Path.cwd()
    hits = list(content.glob("epq3/src/med_doc/__init__.py"))
    hits += list(content.glob("*/src/med_doc/__init__.py"))
    if not hits:
        raise ModuleNotFoundError(
            "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V7 "
            "and combo_backend in run_blocks_1_to_5. Open Tick_Pipeline_Colab.ipynb "
            "from GitHub branch block1. Runtime → Disconnect and delete runtime, then Run all."
        )
    src = hits[0].parents[1]
    root = hits[0].parents[2]
    os.chdir(root)
    sp = str(src.resolve())
    if sp not in sys.path:
        sys.path.insert(0, sp)
    return sp
_guard()

import json
import cv2
import matplotlib.pyplot as plt
import numpy as np

try:
    from google.colab import files as colab_files
except Exception:
    colab_files = None

OUT = Path("/content/tick_pipeline") if Path("/content").is_dir() else Path("outputs/tick_pipeline")
OUT.mkdir(parents=True, exist_ok=True)

def show_rgb(path, title="", figsize=(10, 8)):
    path = Path(path)
    if not path.exists():
        print("missing", path)
        return
    bgr = cv2.imread(str(path))
    if bgr is None:
        print("unreadable", path)
        return
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=figsize)
    plt.imshow(rgb)
    plt.title(title or path.name)
    plt.axis("off")
    plt.show()

def download(path):
    path = Path(path)
    print(path, f"({path.stat().st_size / 1024:.1f} KB)" if path.exists() else "missing")
    if colab_files and path.exists():
        colab_files.download(str(path))

def demo_ticked_sheet(path: Path) -> Path:
    """Synthetic v0 form with slashes on CBC + lipid profile (no PHI)."""
    from med_doc.eval.photoreal import draw_ticks
    from med_doc.template import load_template

    template = load_template()
    w, h = template.width, template.height
    img = np.full((h, w, 3), 255, dtype=np.uint8)
    cv2.rectangle(img, (int(0.02 * w), int(0.02 * h)), (int(0.98 * w), int(0.07 * h)), (30, 30, 30), -1)
    for spec in template.checkbox_fields():
        x0, y0, x1, y1 = template.pixel_bbox(spec, apply_pad=False)
        cv2.rectangle(img, (x0, y0), (x1, y1), (150, 150, 150), 2)
    boxes = []
    fmap = {s.field_id: s for s in template.checkbox_fields()}
    for fid in ("cbc", "profile_lipid"):
        spec = fmap.get(fid)
        if spec is None:
            continue
        boxes.append(list(template.pixel_bbox(spec, apply_pad=False)))
    img = draw_ticks(img, boxes, kind="slash")
    path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(path), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
    print("wrote", path, "ticks", ["cbc", "profile_lipid"])
    return path


## 1. Knobs + batch

`COMBO_BACKEND="kg"` works offline. `"ollama"` needs Ollama on the runtime (typical lab PC, not stock Colab).


In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    content = Path("/content") if Path("/content").is_dir() else Path.cwd()
    hits = list(content.glob("epq3/src/med_doc/__init__.py"))
    hits += list(content.glob("*/src/med_doc/__init__.py"))
    if not hits:
        raise ModuleNotFoundError(
            "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V7 "
            "and combo_backend in run_blocks_1_to_5. Open Tick_Pipeline_Colab.ipynb "
            "from GitHub branch block1. Runtime → Disconnect and delete runtime, then Run all."
        )
    src = hits[0].parents[1]
    root = hits[0].parents[2]
    os.chdir(root)
    sp = str(src.resolve())
    if sp not in sys.path:
        sys.path.insert(0, sp)
    return sp
_guard()

import shutil
from pathlib import Path

USE_UPLOAD = False  # True → pick a photo ZIP; do not upload clinic PHI
OUTPUT_MODE = "dev"
HTR_MODE = "nonverbal"  # skip 3b handwriting
COMBO_BACKEND = "kg"  # placeholder co-occurrence (profile components). "ollama" if you installed it.
COMBO_MODEL = "llama3.2:3b-instruct"
VISION_BACKEND = "off"  # Colab has no Ollama VL by default
BATCH_DIR = None

if USE_UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    names = list(uploaded.keys())
    if len(names) == 1 and names[0].lower().endswith(".zip"):
        batch_input = names[0]
    else:
        batch_input = names
elif BATCH_DIR is not None:
    batch_input = BATCH_DIR
else:
    sheet = demo_ticked_sheet(OUT / "raw" / "lipid_cbc.png")
    raw = OUT / "raw_batch"
    raw.mkdir(parents=True, exist_ok=True)
    shutil.copy2(sheet, raw / "form_a.png")
    batch_input = raw

print("batch_input:", batch_input)
print("htr_mode", HTR_MODE, "combo", COMBO_BACKEND, "vision", VISION_BACKEND)


## 2. Run Blocks 1–5 (tick-oriented)


In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    content = Path("/content") if Path("/content").is_dir() else Path.cwd()
    hits = list(content.glob("epq3/src/med_doc/__init__.py"))
    hits += list(content.glob("*/src/med_doc/__init__.py"))
    if not hits:
        raise ModuleNotFoundError(
            "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V7 "
            "and combo_backend in run_blocks_1_to_5. Open Tick_Pipeline_Colab.ipynb "
            "from GitHub branch block1. Runtime → Disconnect and delete runtime, then Run all."
        )
    src = hits[0].parents[1]
    root = hits[0].parents[2]
    os.chdir(root)
    sp = str(src.resolve())
    if sp not in sys.path:
        sys.path.insert(0, sp)
    return sp
_guard()

from med_doc.pipeline import run_blocks_1_to_5

pipe = run_blocks_1_to_5(
    batch_input,
    output_dir=OUT,
    backend="lexicon",
    patch_missing_edta=False,
    output_mode=OUTPUT_MODE,
    mark_backend="geometry",
    htr_mode=HTR_MODE,
    combo_backend=COMBO_BACKEND,
    combo_model=COMBO_MODEL,
    vision_backend=VISION_BACKEND,
)
print("mode", pipe["output_mode"], "docs", pipe["block5"]["manifest"]["total_documents"])
print("output_json", pipe.get("output_json"))
for row in pipe["block5"]["manifest"]["documents"]:
    print(row["doc_id"], "ticked", row.get("n_ticked"), "hitl", row.get("n_hitl"),
          "needs_review", row.get("needs_review"))


## 3. Ticks, combination flags, overlay

Look for warnings `Combination missing_likely:` (e.g. HDL/LDL/trig if lipid profile ticked) and `hitl_fields`. Those ids were flagged for a second 3a look, not auto-ordered.


In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    content = Path("/content") if Path("/content").is_dir() else Path.cwd()
    hits = list(content.glob("epq3/src/med_doc/__init__.py"))
    hits += list(content.glob("*/src/med_doc/__init__.py"))
    if not hits:
        raise ModuleNotFoundError(
            "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V7 "
            "and combo_backend in run_blocks_1_to_5. Open Tick_Pipeline_Colab.ipynb "
            "from GitHub branch block1. Runtime → Disconnect and delete runtime, then Run all."
        )
    src = hits[0].parents[1]
    root = hits[0].parents[2]
    os.chdir(root)
    sp = str(src.resolve())
    if sp not in sys.path:
        sys.path.insert(0, sp)
    return sp
_guard()

import json
from pathlib import Path

print("=== order.json ===")
bundle = json.loads(Path(pipe["output_json"]).read_text())
for order in bundle.get("orders") or []:
    print("doc", order.get("doc_id"))
    print("  ticked", order.get("ticked_test_ids"))
    print("  implied", order.get("implied_tests"))
    print("  needs_review", order.get("needs_review"))
    print("  warnings (combo):")
    for w in order.get("warnings") or []:
        if "Combination" in w or "Vision tick" in w:
            print("   ", w)

if OUTPUT_MODE == "dev":
    b4 = Path(pipe["block4"]["output_dir"])
    for pred_path in sorted(b4.glob("docs/*/prediction.json")):
        pred = json.loads(pred_path.read_text())
        print("===", pred_path.parent.name, "prediction ===")
        print("  ticked", pred.get("ticked_test_ids"))
        print("  hitl", pred.get("hitl_fields"))
        combo = [w for w in pred.get("warnings") or [] if w.startswith("Combination")]
        print("  combination flags", combo)
        canvas = pred_path.parent / "annotated_canvas.png"
        if canvas.exists():
            show_rgb(canvas, title=pred_path.parent.name)


## 4. Download ZIPs / order.json


In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    content = Path("/content") if Path("/content").is_dir() else Path.cwd()
    hits = list(content.glob("epq3/src/med_doc/__init__.py"))
    hits += list(content.glob("*/src/med_doc/__init__.py"))
    if not hits:
        raise ModuleNotFoundError(
            "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V7 "
            "and combo_backend in run_blocks_1_to_5. Open Tick_Pipeline_Colab.ipynb "
            "from GitHub branch block1. Runtime → Disconnect and delete runtime, then Run all."
        )
    src = hits[0].parents[1]
    root = hits[0].parents[2]
    os.chdir(root)
    sp = str(src.resolve())
    if sp not in sys.path:
        sys.path.insert(0, sp)
    return sp
_guard()

if OUTPUT_MODE == "dev":
    for key in ("block1", "block3", "block4", "block5"):
        z = (pipe.get("zips") or {}).get(key)
        if z:
            download(z)
download(pipe.get("output_json"))
